<a target="_blank" href="https://colab.research.google.com/github/adamya-singh/machine-learning-exercises/blob/master/neuroscope-neel-nanda/Interactive_Neuroscope.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Interactive Neuroscope

*This is an interactive accompaniment to [neuroscope.io](https://neuroscope.io) and to the [studying learned language features post](https://www.alignmentforum.org/posts/Qup9gorqpd9qKAEav/200-cop-in-mi-studying-learned-features-in-language-models) in [200 Concrete Open Problems in Mechanistic Interpretability](https://neelnanda.io/concrete-open-problems)*

There's a surprisingly rich ecosystem of easy ways to create interactive graphics, especially for ML systems. If you're trying to do mechanistic interpretability, the ability to do web dev and to both visualize data and interact with it seems high value!

This version swaps the original dense GPT-2 neuron viewer for an OLMoE expert-routing viewer. The goal is still to show what quickly hacking together a custom visualisation can look like, while keeping the code simple enough to edit in a notebook.

Note that you'll need to run the code yourself to get the interactive interface, so the cell at the bottom will be blank at first!

To emphasise - the point of this notebook is to be a rough proof of concept that just about works, *not* to be the well executed ideal of interactively studying expert routing.

## Setup

In [1]:
# NBVAL_IGNORE_OUTPUT
# Janky code to do different setup when run in a Colab notebook vs VSCode
import os

DEVELOPMENT_MODE = True
IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"
try:
    import google.colab

    IN_COLAB = True
    print("Running as a Colab notebook")
except:
    IN_COLAB = False
    print("Running as a Jupyter notebook - intended for development only!")
    from IPython import get_ipython

    ipython = get_ipython()
    # Code to automatically update notebook imports as they are edited without restarting the kernel
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

if IN_COLAB or IN_GITHUB:
    %pip install "transformers>=4.57.0"
    %pip install torch
    %pip install gradio
    %pip install sentencepiece


Running as a Colab notebook


In [2]:
# NBVAL_IGNORE_OUTPUT
import html

import gradio as gr
import torch
from IPython.display import HTML, display
from transformers import AutoTokenizer, OlmoeForCausalLM


## Extracting Router Probabilities

We first write some code to extract OLMoE router probabilities for all experts at a given layer, for a given text.

In [3]:
# NBVAL_IGNORE_OUTPUT
model_name = "allenai/OLMoE-1B-7B-0924"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = OlmoeForCausalLM.from_pretrained(model_name, dtype="auto").to(device)
model.eval()
print(f"Loaded {model_name} on {device}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

Loaded allenai/OLMoE-1B-7B-0924 on cuda


In [4]:
def get_router_probs(text, layer):
    if layer is None:
        raise ValueError("Please select a layer.")

    layer = int(layer)
    num_layers = model.config.num_hidden_layers

    if not 0 <= layer < num_layers:
        raise ValueError(f"Layer must be between 0 and {num_layers - 1}.")

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_router_logits=True)

    router_logits = outputs.router_logits
    if router_logits is None:
        raise ValueError("Model did not return router logits.")

    layer_router_logits = router_logits[layer]
    if layer_router_logits.dim() == 2:
        batch_size, seq_len = inputs["input_ids"].shape
        layer_router_logits = layer_router_logits.view(batch_size, seq_len, -1)
    elif layer_router_logits.dim() != 3:
        raise ValueError(
            f"Unexpected router logits shape for layer {layer}: {tuple(layer_router_logits.shape)}"
        )

    router_probs = torch.softmax(layer_router_logits.float(), dim=-1)[0]
    input_ids = inputs["input_ids"][0].detach().cpu().tolist()
    str_tokens = tokenizer.convert_ids_to_tokens(input_ids)
    return router_probs.detach().to(torch.float32).cpu().numpy(), str_tokens


We can run this function and verify that it gives a token-by-expert matrix of router probabilities.

In [5]:
default_layer = 0
default_text = "The Apollo program landed astronauts on the moon in 1969."
print(tokenizer.convert_ids_to_tokens(tokenizer(default_text, return_tensors="pt")["input_ids"][0]))


['The', 'ĠApollo', 'Ġprogram', 'Ġlanded', 'Ġastronaut', 's', 'Ġon', 'Ġthe', 'Ġmoon', 'Ġin', 'Ġ1969', '.']


In [6]:
# NBVAL_IGNORE_OUTPUT
router_probs, tokens = get_router_probs(default_text, default_layer)
print(router_probs.shape)
print(router_probs[0])
print(router_probs.sum(axis=1))


(12, 64)
[0.0157381  0.0155162  0.01561265 0.01556145 0.01559487 0.01822436
 0.01600822 0.01544171 0.01543373 0.01567007 0.01552175 0.01538943
 0.01552751 0.01540374 0.01694022 0.01558856 0.01542562 0.01574312
 0.01619197 0.01580323 0.01538482 0.01549471 0.01541166 0.01555437
 0.01542342 0.01551784 0.0154911  0.01558773 0.01553108 0.01546525
 0.01555094 0.01558083 0.01552929 0.01542306 0.01541804 0.01549796
 0.01550778 0.01555577 0.01599578 0.01556584 0.01544968 0.01574675
 0.01546322 0.01552988 0.01549334 0.01570941 0.01540675 0.01549987
 0.01554103 0.01553169 0.01551192 0.01551088 0.01549796 0.01540333
 0.01548588 0.01558163 0.01554039 0.01539059 0.01542342 0.01545555
 0.01546875 0.01651508 0.01556363 0.01545555]
[1.         1.         1.         1.         1.         1.
 0.99999994 0.99999994 1.         1.         1.         0.9999999 ]


## Visualizing Router Probabilities

We now visualize router probabilities for all experts at one selected layer. The display is a simple token-by-expert heatmap built in raw HTML so it stays easy to hack on inside the notebook.

Each column is a token, each row is an expert, and each cell color shows that expert's routing probability for that token.

In [7]:
# This is some CSS to display a compact token-by-expert heatmap.
style_string = """<style>
    table.router-heatmap {
        border-collapse: collapse;
        font-family: monospace;
        font-size: 12px;
    }
    table.router-heatmap th, table.router-heatmap td {
        border: 1px solid rgb(123, 123, 123);
        padding: 4px;
        text-align: center;
        min-width: 48px;
    }
    table.router-heatmap th.token-header {
        writing-mode: vertical-rl;
        transform: rotate(180deg);
        white-space: pre-wrap;
    }
    table.router-heatmap th.expert-label {
        position: sticky;
        left: 0;
        background: white;
    }
    div.router-scroll {
        overflow-x: auto;
        max-width: 100%;
    }
    </style>"""


def calculate_color(val, max_val, min_val):
    # Normalize to [0, 1] and interpolate between slightly off-white and red.
    denom = max(max_val - min_val, 1e-8)
    normalized_val = min(max((val - min_val) / denom, 0.0), 1.0)
    return f"rgb(240, {240 * (1 - normalized_val)}, {240 * (1 - normalized_val)})"


def basic_router_vis(text, layer, max_val=None, min_val=None):
    """
    text: The text to visualize
    layer: The layer index
    max_val: The top end of our probability range, defaults to the maximum probability
    min_val: The bottom end of our probability range, defaults to the minimum probability

    Returns a string of HTML that displays a token-by-expert router probability heatmap.
    """
    try:
        probs, str_tokens = get_router_probs(text, layer)
    except Exception as error:
        return str(error)

    prob_max = probs.max()
    prob_min = probs.min()
    if max_val is None:
        max_val = prob_max
    if min_val is None:
        min_val = prob_min

    htmls = [style_string]
    htmls.append(f"<h4>Layer: <b>{int(layer)}</b></h4>")
    htmls.append(
        f"<h4>Display Range: <b>{max_val:.4f}</b>. Min Range: <b>{min_val:.4f}</b></h4>"
    )
    htmls.append(
        f"<h4>Actual Probabilities: <b>{prob_min:.4f}</b> to <b>{prob_max:.4f}</b></h4>"
    )

    htmls.append("<div class='router-scroll'><table class='router-heatmap'>")
    htmls.append("<tr><th>Expert</th>")
    for tok in str_tokens:
        safe_tok = html.escape(tok)
        htmls.append(f"<th class='token-header'>{safe_tok}</th>")
    htmls.append("</tr>")

    num_experts = probs.shape[1]
    for expert_idx in range(num_experts):
        htmls.append(f"<tr><th class='expert-label'>E{expert_idx}</th>")
        for token_idx in range(len(str_tokens)):
            prob = probs[token_idx, expert_idx]
            color = calculate_color(prob, max_val, min_val)
            htmls.append(
                f"<td style='background-color:{color}' title='p={prob:.6f}'>{prob:.4f}</td>"
            )
        htmls.append("</tr>")

    htmls.append("</table></div>")
    return "".join(htmls)


In [8]:
# NBVAL_IGNORE_OUTPUT
# The function outputs a string of HTML
default_max_val = 1.0
default_min_val = 0.0
default_html_string = basic_router_vis(
    default_text,
    default_layer,
    max_val=default_max_val,
    min_val=default_min_val,
)

# IPython lets us display HTML
print("Displayed HTML")
display(HTML(default_html_string))

# We can also print the string directly
print("HTML String - it's just raw HTML code!")
print(default_html_string)


Displayed HTML


Expert,The,ĠApollo,Ġprogram,Ġlanded,Ġastronaut,s,Ġon,Ġthe,Ġmoon,Ġin,Ġ1969,.
E0,0.0157,0.0156,0.0154,0.0155,0.0155,0.0154,0.0154,0.0155,0.0155,0.0155,0.0155,0.0155
E1,0.0155,0.0156,0.0156,0.0156,0.0156,0.0155,0.0156,0.0155,0.0156,0.0156,0.0156,0.0155
E2,0.0156,0.0156,0.0156,0.0155,0.0157,0.0156,0.0156,0.0156,0.0157,0.0156,0.0156,0.0156
E3,0.0156,0.0155,0.0159,0.0155,0.0156,0.0155,0.0158,0.0157,0.0156,0.0156,0.0155,0.0156
E4,0.0156,0.0155,0.0157,0.0155,0.0156,0.0155,0.0156,0.0155,0.0156,0.0155,0.0156,0.0155
E5,0.0182,0.0183,0.0156,0.0156,0.0155,0.0155,0.0155,0.0156,0.0162,0.0159,0.0155,0.0156
E6,0.0160,0.0159,0.0158,0.0158,0.0157,0.0157,0.0160,0.0165,0.0162,0.0162,0.0162,0.0164
E7,0.0154,0.0154,0.0155,0.0154,0.0154,0.0168,0.0156,0.0156,0.0154,0.0156,0.0154,0.0157
E8,0.0154,0.0157,0.0155,0.0155,0.0155,0.0155,0.0156,0.0156,0.0156,0.0157,0.0156,0.0157
E9,0.0157,0.0155,0.0154,0.0154,0.0156,0.0154,0.0154,0.0154,0.0155,0.0154,0.0157,0.0154


HTML String - it's just raw HTML code!
<style>
    table.router-heatmap {
        border-collapse: collapse;
        font-family: monospace;
        font-size: 12px;
    }
    table.router-heatmap th, table.router-heatmap td {
        border: 1px solid rgb(123, 123, 123);
        padding: 4px;
        text-align: center;
        min-width: 48px;
    }
    table.router-heatmap th.token-header {
        writing-mode: vertical-rl;
        transform: rotate(180deg);
        white-space: pre-wrap;
    }
    table.router-heatmap th.expert-label {
        position: sticky;
        left: 0;
        background: white;
    }
    div.router-scroll {
        overflow-x: auto;
        max-width: 100%;
    }
    </style><h4>Layer: <b>0</b></h4><h4>Display Range: <b>1.0000</b>. Min Range: <b>0.0000</b></h4><h4>Actual Probabilities: <b>0.0154</b> to <b>0.0183</b></h4><div class='router-scroll'><table class='router-heatmap'><tr><th>Expert</th><th class='token-header'>The</th><th class='token-header'>ĠA

## Create Interactive UI

We now put all these together to create an interactive visualization in Gradio.

The interface now shows all experts for one selected layer at a time as a token-by-expert heatmap.

In [9]:
# The `with gr.Blocks() as demo:` syntax just creates a variable called demo containing all these components
with gr.Blocks() as demo:
    gr.HTML(value=f"Hacky Interactive OLMoE Router Heatmap for {model_name}")
    # The input elements
    with gr.Row():
        with gr.Column():
            text = gr.Textbox(label="Text", value=default_text)
            # Precision=0 makes it an int, otherwise it's a float
            # Value sets the initial default value
            layer = gr.Number(label="Layer", value=default_layer, precision=0)
            # If empty, these two map to None
            max_val = gr.Number(label="Max Value", value=default_max_val)
            min_val = gr.Number(label="Min Value", value=default_min_val)
            inputs = [text, layer, max_val, min_val]
        with gr.Column():
            # The output element
            out = gr.HTML(label="Router Probability Heatmap", value=default_html_string)
    for inp in inputs:
        inp.change(basic_router_vis, inputs, out)


We can now launch our demo element, and we're done! The setting share=True even gives you a public link to the demo (though it just redirects to the backend run by this notebook, and will go away once you turn the notebook off!). Sharing makes it much slower, and can be turned off if you aren't in Colab.

**Exercise:** Explore how the full expert distribution changes across layers and across different texts. Which tokens concentrate mass on a small set of experts, and which spread it across many experts?

In [10]:
# NBVAL_IGNORE_OUTPUT
demo.launch(share=True, height=1000)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ce8d36be328cbee7ef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
